In [1]:
%matplotlib inline
%load_ext Cython

In [2]:
import math
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from numba import jit, vectorize, float64, int64

In [3]:
sns.set_context('notebook', font_scale=1.5)

Making Python faster

In this quiz, you will practice making Python code faster. We start with functions that already use `numpy` (which are about two orders of magnitude faster than the pure Python versions).

**Functions to optimize**

In [4]:
def logistic(x):
    """Logistic function."""

    # UPDATED FUNCTION to only call np.exp once
    e = np.exp(x)
    return e/(1 + e)

def gd(X, y, beta, alpha, niter):
    """Gradient descent algorihtm."""
    n, p = X.shape
    Xt = X.T
    for i in range(niter):
        y_pred = logistic(X @ beta)
        epsilon = y - y_pred
        grad = Xt @ epsilon / n
        beta += alpha * grad
    return beta

In [5]:
x = np.linspace(-6, 6, 100)
plt.plot(x, logistic(x))
pass

**Data set for classification**

In [6]:
n = 10000
p = 2
X, y = make_blobs(n_samples=n, n_features=p, centers=2, cluster_std=1.05, random_state=23)
X = np.c_[np.ones(len(X)), X]
y = y.astype('float')

**Using gradient descent for classification by logistic regression**

In [7]:
# initial parameters
niter = 1000
α = 0.01
β = np.zeros(p+1)

# call gradient descent
β = gd(X, y, β, α, niter)

# assign labels to points based on prediction
y_pred = logistic(X @ β)
labels = y_pred > 0.5

# calculate separating plane
sep = (-β[0] - β[1] * X)/β[2]

plt.scatter(X[:, 1], X[:, 2], c=labels, cmap='winter')
plt.plot(X, sep, 'r-')
pass

**1**. Rewrite the `logistic` function so it only makes one `np.exp` call. Compare the time of both versions with the input x given below using the `@timeit`. (3 points)

In [8]:
np.random.seed(123)
n = int(1e7)
x = np.random.normal(0, 1, n)

In [9]:
%timeit logistic(x)




60 ms ± 1.46 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


**2**. (7 points) Use `numba` to compile the gradient descent function.

- Use the `@vectorize` decorator to create a ufunc version of the logistic function and call this `logistic_numba_cpu` with function signatures of `float64(float64)`. Create another function called `logistic_numba_parallel` by giving an extra argument to the decorator of `target=parallel` (2 points)
- For each function, check that the answers are the same as with the original logistic function using  `np.testing.assert_array_almost_equal`. Use `%timeit` to compare the three logistic functions (2 points)
- Now use `@jit` to create a JIT_compiled version of the `logistic` and `gd` functions, calling them `logistic_numba` and `gd_numba`. Provide appropriate function signatures to the decorator in each case. (2 points)
- Compare the two gradient descent functions `gd` and `gd_numba` for correctness and performance. (1 points)

In [10]:
# Step 1a: Create CPU vectorized version
@vectorize([float64(float64)])
def logistic_numba_cpu(x):
    """Logistic function - Numba CPU vectorized version."""
    e = np.exp(x)
    return e / (1 + e)

# Step 1b: Create parallel vectorized version
@vectorize([float64(float64)], target='parallel')
def logistic_numba_parallel(x):
    """Logistic function - Numba parallel vectorized version."""
    e = np.exp(x)
    return e / (1 + e)


In [11]:
# Verify all three functions give same results
original_result = logistic(x)
np.testing.assert_array_almost_equal(logistic_numba_cpu(x), original_result)
np.testing.assert_array_almost_equal(logistic_numba_parallel(x), original_result)

print("All logistic functions produce identical results")

All logistic functions produce identical results


In [12]:
# Original numpy logistic
%timeit logistic(x)

59.6 ms ± 805 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [13]:
# Numba CPU vectorized
%timeit logistic_numba_cpu(x)

47.9 ms ± 521 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
# Numba parallel vectorized
%timeit logistic_numba_parallel(x)

5.77 ms ± 140 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [23]:
# JIT-compiled logistic function
@jit(float64[:](float64[:]), nopython=True, cache=True,)
def logistic_numba(x):
    """Logistic function - JIT compiled version."""
    e = np.exp(x)
    return e / (1 + e)

# JIT-compiled gradient descent function
@jit(float64[:](float64[:,:], float64[:], float64[:], float64, int64), nopython=True, parallel=True, cache=True)
def gd_numba(X, y, beta, alpha, niter):
    """Gradient descent algorithm - JIT compiled version."""
    n, p = X.shape
    Xt = X.T
    for i in range(niter):
        y_pred = logistic_numba(X @ beta)
        epsilon = y - y_pred
        grad = Xt @ epsilon / n
        beta += alpha * grad
    return beta

/var/folders/hl/18z9x6cd0cs071nkgz798m6h0000gn/T/ipykernel_97996/31027232.py:15: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 1, 'A', False, aligned=True))
  y_pred = logistic_numba(X @ beta)
/var/folders/hl/18z9x6cd0cs071nkgz798m6h0000gn/T/ipykernel_97996/31027232.py:17: NumbaPerformanceWarning: '@' is faster on contiguous arrays, called on (Array(float64, 2, 'A', False, aligned=True), Array(float64, 1, 'C', False, aligned=True))
  grad = Xt @ epsilon / n


In [24]:
# Reset parameters for fair comparison
β_original = np.zeros(p+1)
β_numba = np.zeros(p+1)

# Run both versions
result_original = gd(X, y, β_original.copy(), α, niter)
result_numba = gd_numba(X, y, β_numba.copy(), α, niter)

# Compare results
np.testing.assert_array_almost_equal(result_original, result_numba, decimal=10)
print("Both gradient descent functions produce identical results")
print(f"\nOriginal gd β: {result_original}")
print(f"Numba gd β:    {result_numba}")

Both gradient descent functions produce identical results

Original gd β: [ 0.04482678  0.45549046 -0.79878438]
Numba gd β:    [ 0.04482678  0.45549046 -0.79878438]


In [25]:
# Time original gradient descent
%timeit gd(X, y, np.zeros(p+1), α, niter)

92.9 ms ± 750 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [26]:
# Time JIT-compiled gradient descent
%timeit gd_numba(X, y, np.zeros(p+1), α, niter)

100 ms ± 750 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Summary


Interestingly, when I added specific parameters like this nopython=True, parallel=True, cache=True the program slowed down. I am not sure why this is the case but interesting area for deeper due diligence.

